# 01 — Discovery (DirectoryAgent + SearchAgent)
Part of the Public Webcam Discovery System.

Runs DirectoryAgent and SearchAgent against Tier 1 sources. Inspect and save candidates.jsonl.

In [ ]:
# ── Environment setup (run once) ─────────────────────────────
import subprocess, sys, os
from pathlib import Path

IN_COLAB     = "google.colab" in sys.modules
IN_SAGEMAKER = os.environ.get("SM_TRAINING_ENV") is not None

if IN_COLAB:
    if not Path("webcam-discovery").exists():
        subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/webcam-discovery"], check=True)
    os.chdir("webcam-discovery")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[notebooks]", "-q"], check=True)

from webcam_discovery.config import settings
settings.ensure_dirs()
print("✓ Ready")

## Step 2 — Build environment

Install the package with dev dependencies so the pipeline and tests are available.

In [ ]:
!pip install -e ".[dev]" -q

## Step 3 — Run DirectoryAgent (Tier 1)

Crawls all Tier 1 sources from `SOURCES.md`, resolves direct feed URLs via
`FeedExtractionSkill`, and writes results to `candidates/candidates.jsonl`.

In [ ]:
!python scripts/run_discovery.py --tier 1 --output candidates/candidates.jsonl

## Step 4 — Inspect candidates.jsonl

Preview the first few results and count totals.

In [ ]:
import json, re
from pathlib import Path
from collections import Counter

output_path = Path("candidates/candidates.jsonl")

if not output_path.exists():
    print("candidates.jsonl not found — did the crawler run successfully?")
else:
    lines = output_path.read_text().splitlines()
    candidates = [json.loads(l) for l in lines if l.strip()]

    _stream_ext   = re.compile(r'\.(m3u8|mjpg|mjpeg|mp4)(\?|$)', re.IGNORECASE)
    _youtube_re   = re.compile(r'youtube(?:-nocookie)?\.com/embed/', re.IGNORECASE)
    _embed_re     = re.compile(r'/embed[/\?]|embed\.|\.embed', re.IGNORECASE)

    def url_type(url):
        if _stream_ext.search(url):  return 'direct-stream'
        if _youtube_re.search(url):  return 'youtube-embed'
        if _embed_re.search(url):    return 'embed-page'
        return 'html-page'

    types = Counter(url_type(c['url']) for c in candidates)

    print(f"Total candidates: {len(candidates)}")
    print(f"  direct-stream  : {types['direct-stream']:>4}  (HLS / MJPEG / MP4 — ready for validation)")
    print(f"  youtube-embed  : {types['youtube-embed']:>4}  (YouTube nocookie embed)")
    print(f"  embed-page     : {types['embed-page']:>4}  (third-party player embed)")
    print(f"  html-page      : {types['html-page']:>4}  (camera embed pages / may need deeper extraction)")
    print()

    # Show first 10 with type annotation
    for c in candidates[:10]:
        t = url_type(c['url'])
        print(f"  [{t:<14}]  {c['url']}")
        print(f"               label={c.get('label','—')}  city={c.get('city','—')}, {c.get('country','—')}")
        print()

## Step 5 — Validate candidates

`ValidationAgent` checks each candidate URL with HTTP HEAD/GET, classifies the
feed type (HLS, MJPEG, YouTube, iframe, …), and geo-enriches with lat/lon.

- **high** legitimacy → direct media stream confirmed live
- **medium** legitimacy → HTML embed page, camera likely accessible
- **low** legitimacy → auth-gated or dead; rejected

Results written to `candidates/validated.jsonl` (one `CameraRecord` per line).

In [ ]:
!python scripts/run_validation.py --input candidates/candidates.jsonl --output candidates/validated.jsonl

## Step 6 — Inspect validated cameras

Review which candidates passed validation, their feed types, and legitimacy scores.

In [ ]:
import json
from pathlib import Path
from collections import Counter

validated_path = Path("candidates/validated.jsonl")
candidates_path = Path("candidates/candidates.jsonl")

if not validated_path.exists():
    print("validated.jsonl not found — did validation run successfully?")
else:
    records = [json.loads(l) for l in validated_path.read_text().splitlines() if l.strip()]

    # Pipeline funnel summary
    n_candidates = sum(1 for _ in candidates_path.read_text().splitlines() if _.strip()) \
                   if candidates_path.exists() else "?"
    print("── Pipeline funnel ──────────────────────────────")
    print(f"  Candidates (discovery)  : {n_candidates}")
    print(f"  Validated cameras       : {len(records)}")
    print()

    # Breakdown by feed type and legitimacy
    feed_types = Counter(r['feed_type'] for r in records)
    legit_scores = Counter(r['legitimacy_score'] for r in records)
    statuses = Counter(r['status'] for r in records)

    print("── By feed type ─────────────────────────────────")
    for ft, n in feed_types.most_common():
        print(f"  {ft:<16} : {n}")

    print()
    print("── By legitimacy score ──────────────────────────")
    for ls, n in legit_scores.most_common():
        print(f"  {ls:<8} : {n}")

    print()
    print("── By status ────────────────────────────────────")
    for st, n in statuses.most_common():
        print(f"  {st:<8} : {n}")

    print()
    print("── First 10 validated cameras ───────────────────")
    for r in records[:10]:
        print(f"  [{r['feed_type']:<14}] [{r['legitimacy_score']:<6}]  {r['url']}")
        print(f"    {r.get('label','—')} — {r.get('city','—')}, {r.get('country','—')}")
        print()